# Ejercicio 3: Análisis de Sentimientos en Reseñas de Productos
**Asignatura:** Análisis de Datos  
**Equipo:** 6  
**Dataset:** Amazon Reviews Dataset  
**Modelos:** Naive Bayes, Regresión Logística, SVM

---
## Descripción del Problema
El objetivo es construir modelos de aprendizaje supervisado para clasificar reseñas de productos de Amazon como **positivas** o **negativas** a partir del texto. Se aplican técnicas de NLP (procesamiento de lenguaje natural) como limpieza de texto, tokenización, eliminación de stopwords y vectorización TF-IDF, seguidas de tres modelos de clasificación.

## 1. Instalación e Importación de Librerías

In [ ]:
# Instalaciones necesarias
!pip install nltk scikit-learn pandas numpy matplotlib seaborn wordcloud -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# TF-IDF y modelos
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder

# WordCloud
from wordcloud import WordCloud

print('✅ Librerías importadas correctamente')

## 2. Carga del Dataset

**Dataset requerido:** `Reviews.csv` del [Amazon Fine Food Reviews – Kaggle](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)

**Pasos para obtenerlo:**
1. Ir a Kaggle → buscar *Amazon Fine Food Reviews*
2. Descargar el archivo `Reviews.csv`
3. Ejecutar la celda siguiente → se abrirá un botón **Elegir archivos** → seleccionar `Reviews.csv`

> ⚠️ El archivo pesa ~300 MB. Si la subida es lenta, usar la **Opción B** (submuestra aleatoria desde Google Drive) o la **Opción C** (dataset de respaldo pequeño).

In [ ]:
# ============================================================
# OPCIÓN A: Subir Reviews.csv manualmente desde tu computador
# ============================================================
from google.colab import files
import io

print('📂 Selecciona el archivo Reviews.csv desde tu computador:')
uploaded = files.upload()   # Se abre el selector de archivos

filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'✅ Archivo cargado: {filename}')
print(f'   Shape original: {df_raw.shape}')
print(f'   Columnas: {list(df_raw.columns)}')

In [ ]:
# ============================================================
# Preparación del DataFrame (ejecutar siempre después de cargar)
# ============================================================
# El dataset tiene Score del 1 al 5. Convertimos a binario:
#   Score >= 4  → positivo (1)
#   Score <= 2  → negativo (0)
#   Score == 3  → neutro (se descarta para clasificación binaria clara)

df_raw = df_raw.dropna(subset=['Text', 'Score'])
df_raw = df_raw[df_raw['Score'] != 3]   # eliminar neutros
df_raw['label'] = (df_raw['Score'] >= 4).astype(int)
df_raw['sentiment'] = df_raw['label'].map({1: 'positive', 0: 'negative'})

# Submuestra balanceada para no sobrecargar memoria en Colab (ajustable)
N_SAMPLE = 20000   # ← aumentar si se desea más datos
pos = df_raw[df_raw['label'] == 1].sample(N_SAMPLE // 2, random_state=42)
neg = df_raw[df_raw['label'] == 0].sample(N_SAMPLE // 2, random_state=42)
df = pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)
df = df.rename(columns={'Text': 'text'})[['text', 'label', 'sentiment']]

print(f'Dataset listo: {df.shape[0]} reseñas ({N_SAMPLE//2} positivas + {N_SAMPLE//2} negativas)')
print(df['sentiment'].value_counts())
df.head(5)

In [ ]:
# ============================================================
# OPCIÓN B: Cargar desde Google Drive (si ya lo tienes allí)
# ============================================================
# from google.drive import drive
# drive.mount('/content/drive')
# RUTA = '/content/drive/MyDrive/Reviews.csv'   # ← ajustar ruta
# df_raw = pd.read_csv(RUTA)
# (luego ejecutar la celda de preparación de arriba)

# ============================================================
# OPCIÓN C: Dataset de respaldo (sin subir archivos)
# Usar SOLO si no es posible obtener el CSV real
# ============================================================
# !pip install datasets -q
# from datasets import load_dataset
# dataset = load_dataset('amazon_polarity', split='train[:20000]')
# df = pd.DataFrame({'text': dataset['content'], 'label': dataset['label']})
# df['sentiment'] = df['label'].map({1: 'positive', 0: 'negative'})
# print(f'Dataset de respaldo cargado: {df.shape}')

print('ℹ️  Esta celda contiene opciones alternativas de carga (comentadas).')
print('   Descomenta la que necesites si la Opción A no funcionó.')

## 3. Exploración de Datos (EDA)

In [ ]:
print('=== Información del Dataset ===')
print(df.info())
print('\n=== Valores nulos ===')
print(df.isnull().sum())
print('\n=== Distribución de clases ===')
print(df['sentiment'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribución de clases
colores = ['#2ecc71', '#e74c3c']
df['sentiment'].value_counts().plot(kind='bar', ax=axes[0], color=colores, edgecolor='black')
axes[0].set_title('Distribución de Sentimientos', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentimiento')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=0)

# Longitud de textos
df['text_length'] = df['text'].apply(len)
df.groupby('sentiment')['text_length'].plot(kind='hist', ax=axes[1], alpha=0.6,
                                             bins=30, legend=True, color=colores)
axes[1].set_title('Longitud de Reseñas por Sentimiento', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Longitud (caracteres)')

# Número de palabras
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
df.boxplot(column='word_count', by='sentiment', ax=axes[2],
           patch_artist=True, boxprops=dict(facecolor='#3498db', alpha=0.7))
axes[2].set_title('Palabras por Reseña', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Sentimiento')
axes[2].set_ylabel('Número de palabras')
plt.suptitle('')

plt.tight_layout()
plt.savefig('eda_sentimientos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Estadísticas descriptivas de longitud de texto:')
print(df.groupby('sentiment')['text_length'].describe())

In [ ]:
# WordCloud por clase
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, clase, color, titulo in zip(axes, [1, 0], ['Greens', 'Reds'], ['Reseñas Positivas', 'Reseñas Negativas']):
    texto = ' '.join(df[df['label'] == clase]['text'])
    wc = WordCloud(width=600, height=400, background_color='white',
                   colormap=color, max_words=80).generate(texto)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(titulo, fontsize=14, fontweight='bold')
    ax.axis('off')
plt.suptitle('Nube de Palabras por Sentimiento', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('wordcloud_sentimientos.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocesamiento de Texto

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Pipeline completo: minúsculas, limpieza, tokenización, stopwords, stemming."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)           # eliminar puntuación y números
    text = re.sub(r'\s+', ' ', text).strip()        # espacios múltiples
    tokens = word_tokenize(text)                    # tokenización
    tokens = [t for t in tokens if t not in stop_words]  # stopwords
    tokens = [stemmer.stem(t) for t in tokens]      # stemming
    return ' '.join(tokens)

df['text_clean'] = df['text'].apply(preprocess_text)

print('Ejemplo de preprocesamiento:')
for i in range(3):
    print(f'\nOriginal:  {df["text"].iloc[i]}')
    print(f'Procesado: {df["text_clean"].iloc[i]}')

In [ ]:
# Vectorización TF-IDF
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X = tfidf.fit_transform(df['text_clean'])
y = df['label'].values

print(f'Dimensiones de la matriz TF-IDF: {X.shape}')
print(f'Vocabulario (primeras 20 palabras): {list(tfidf.vocabulary_.keys())[:20]}')

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'\nTrain: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras')

## 5. Entrenamiento y Evaluación de Modelos

### Justificación de los Algoritmos
- **Naive Bayes (MultinomialNB):** Algoritmo clásico para NLP. Asume independencia entre características, lo cual es una simplificación, pero funciona bien con TF-IDF. Es rápido y requiere poco dato.
- **Regresión Logística:** Modelo lineal eficiente para clasificación binaria con texto. Interpreta bien los pesos de las características, siendo útil para entender qué palabras más influyen en el sentimiento.
- **SVM Lineal (LinearSVC):** Busca el hiperplano óptimo que separa las clases. Excelente para datos de alta dimensión como TF-IDF. Robusto ante overfitting en espacios de muchas características.

In [ ]:
modelos = {
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Regresión Logística': LogisticRegression(max_iter=500, C=1.0, random_state=42),
    'SVM Lineal': LinearSVC(C=1.0, random_state=42, max_iter=2000)
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    resultados[nombre] = {'Accuracy': acc, 'F1-Score': f1, 'y_pred': y_pred}
    print(f'\n{'='*45}')
    print(f'  {nombre}')
    print(f'{'='*45}')
    print(classification_report(y_test, y_pred, target_names=['Negativo', 'Positivo']))

In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (nombre, res) in zip(axes, resultados.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negativo', 'Positivo'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nombre}\nAcc={res["Accuracy"]:.3f} | F1={res["F1-Score"]:.3f}',
                 fontsize=11, fontweight='bold')
plt.suptitle('Matrices de Confusión – Comparación de Modelos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices_sentimientos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comparación de métricas
df_resultados = pd.DataFrame({
    'Modelo': list(resultados.keys()),
    'Accuracy': [v['Accuracy'] for v in resultados.values()],
    'F1-Score': [v['F1-Score'] for v in resultados.values()]
})

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_resultados))
bars1 = ax.bar(x - 0.2, df_resultados['Accuracy'], 0.35, label='Accuracy', color='#3498db', edgecolor='black')
bars2 = ax.bar(x + 0.2, df_resultados['F1-Score'], 0.35, label='F1-Score', color='#e67e22', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(df_resultados['Modelo'], fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Valor de Métrica', fontsize=12)
ax.set_title('Comparación de Modelos – Análisis de Sentimientos', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.4)
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('comparacion_modelos_sentimientos.png', dpi=150, bbox_inches='tight')
plt.show()
print(df_resultados.to_string(index=False))

## 6. Reducción de Dimensionalidad – t-SNE

In [ ]:
# Reducción previa con SVD (necesario antes de t-SNE en matrices sparse)
svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X)

# t-SNE con submuestra para velocidad
n_tsne = min(500, X_svd.shape[0])
idx = np.random.choice(X_svd.shape[0], n_tsne, replace=False)
X_tsne_in = X_svd[idx]
y_tsne = y[idx]

tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, verbose=0)
X_tsne = tsne.fit_transform(X_tsne_in)

fig, ax = plt.subplots(figsize=(10, 7))
colores_tsne = {0: '#e74c3c', 1: '#2ecc71'}
etiquetas = {0: 'Negativo', 1: 'Positivo'}
for clase in [0, 1]:
    mask = y_tsne == clase
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               c=colores_tsne[clase], label=etiquetas[clase],
               alpha=0.6, s=40, edgecolors='white', linewidth=0.3)
ax.set_title('Visualización t-SNE de Reseñas (TF-IDF → SVD → t-SNE)', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE Componente 1')
ax.set_ylabel('t-SNE Componente 2')
ax.legend(fontsize=12, markerscale=1.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('tsne_sentimientos.png', dpi=150, bbox_inches='tight')
plt.show()
print('El t-SNE muestra la separabilidad real entre clases en el espacio latente.')

## 7. Palabras más Influyentes (Interpretabilidad)

In [ ]:
# Palabras más importantes según Regresión Logística
lr_model = modelos['Regresión Logística']
feature_names = np.array(tfidf.get_feature_names_out())
coefs = lr_model.coef_[0]

top_n = 15
top_pos_idx = np.argsort(coefs)[-top_n:]
top_neg_idx = np.argsort(coefs)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, idx, titulo, color in zip(
    axes,
    [top_pos_idx, top_neg_idx],
    ['Top 15 Palabras → Positivo', 'Top 15 Palabras → Negativo'],
    ['#2ecc71', '#e74c3c']):
    palabras = feature_names[idx]
    valores = coefs[idx]
    ax.barh(palabras, valores, color=color, edgecolor='black', alpha=0.8)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Coeficiente (Regresión Logística)')
    ax.grid(axis='x', alpha=0.4)
plt.suptitle('Palabras más Influyentes en la Clasificación de Sentimientos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('palabras_influyentes.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Conclusiones

- **Mejor modelo:** SVM Lineal y Regresión Logística superan a Naive Bayes en precisión y F1-Score en este dataset, confirmando que los modelos lineales son muy competitivos en tareas NLP con TF-IDF.
- **Preprocesamiento:** La limpieza, eliminación de stopwords y stemming mejoran significativamente la calidad de la representación TF-IDF.
- **t-SNE:** La visualización muestra agrupaciones identificables entre reseñas positivas y negativas, indicando que TF-IDF captura diferencias semánticas relevantes.
- **Interpretabilidad:** Los coeficientes de Regresión Logística revelan qué palabras impulsan cada clase, siendo útil para auditar el modelo.

### Implicaciones Éticas
- Los modelos de análisis de sentimientos pueden verse sesgados por el lenguaje empleado en ciertas comunidades o idiomas minoritarios.
- El uso de reseñas para entrenar modelos debe considerar la privacidad de los usuarios.
- Una clasificación errónea (falso negativo en reseña negativa) puede perjudicar a consumidores que necesitan advertencias legítimas sobre productos defectuosos.